<a href="https://colab.research.google.com/github/jashwanth-cse/Jashwanth-Codeboosters-Internship-2026/blob/main/Phase_02_GenAI_with_Cloud_APIs/Day_08_RAG_Systems/Day08_RAG_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RAG
* Level-1 Simple English
* Level-2 Analogy
* Level-3 Technical

How ChromaDB retreives:
1. User asks a question
2. Question is converted to vector
3. ChromaDB compares query vector against all stored vectors
4. Returns top k most similar documents
5. Sorted by distance(lower=more similar)

In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

In [ ]:
import pandas as pd
import chromadb
import os
from groq import Groq
from sentence_transformers import SentenceTransformer



In [ ]:
GROK_API_KEY="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"
os.environ["GROK_API_KEY"]=GROK_API_KEY
groq_client=Groq(api_key=GROK_API_KEY)

In [ ]:
import sys
if 'pandas' in sys.modules:
    del sys.modules['pandas']
import pandas as pd
df=pd.read_csv("/content/college_notes.csv")

print("Shape of dataset",df.shape)

print("Column names",df.columns.tolist())


Shape of dataset (15, 4)
Column names ['note_id', 'subject', 'topic', 'content']


In [ ]:
print("Subjects in the dataset")
print(df['subject'].value_counts())


Subjects in the dataset
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64


In [ ]:
documents=df['content'].tolist()

ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]

print("Total chunks prepared:",len(documents))
print("First document ID:",ids[0])
print("First metadata",metadatas[0])
print("First 100 chars of doc:",documents[0][:100])

Total chunks prepared: 15
First document ID: note_N001
First metadata {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc


In [ ]:
print("Loading embedding model")
print("This may take some time for download")

embedding_model=SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded")

test_embedding=embedding_model.encode("This is a test sentence")
print("Test embedding shape",test_embedding.shape)

Loading embedding model
This may take some time for download


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded
Test embedding shape (384,)


In [ ]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="college_notes")
print("Collection name",collection.name)
print("Number of documents in collection",collection.count())

Collection name college_notes
Number of documents in collection 0


In [ ]:
print("Generating embeddings for all 15 notes")

embeddings= embedding_model.encode(documents,show_progress_bar=True)
print("Embeddings shape",embeddings.shape)
print(collection.count())


Generating embeddings for all 15 notes


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape (15, 384)
0


In [ ]:
def retrieve_relavent_chunk(question,top_k=3):
    question_embedding=embedding_model.encode(question).tolist()
    results=collection.query(
        query_embeddings=question_embedding,
        n_results=top_k,
    )
    return results
print("retreival function defined successfully")
print("Function:retreive_relevent_chunks")
test_question="What is ETL and how does it work in data engineering?"
print(f"Test Quetion: {test_question}")
print("="*60)
results=retrieve_relavent_chunk(test_question,top_k=3)
print("\nTop 3 relavent chunks")
print("="*60)
for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
  print(f"\nResult {i+1}:")
  print(f"  Subject  : {meta['subject']}")


retreival function defined successfully
Function:retreive_relevent_chunks
Test Quetion: What is ETL and how does it work in data engineering?

Top 3 relavent chunks


Context Injection:


RAG Prompt template

System:

User:

Context:

Question:

In [ ]:
def build_context_from_results(results):
    context_parts=[]
    for i,(doc,meta) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0]
    )):
        chunk_test=f"Source{i+1}:{meta['subject']}-{meta['topic']}"
        context_parts
    context_str="\n\n--\n".join(context_parts)
    return context_str
context=build_context_from_results(results)
print(context)

In [ ]:
def generate_rag_answer(question,context):
  system_prompt="""You are helpful academic assistant for engineering students.
  You will be given context retrieved from a college knowledge base, and a student's question.

  RULES:
  1.Answer only using the information provided in the context below.
  2.If the answer is not found in the context, say exactly:
  "I dont have enough information in my knowledge base to answer this question."
  3.Do not use your general training knowledge.
  4.Keep answers clear, accurate, and beginner-friendly.
  5.Mention which source the information came from when possible."""
  user_prompt=f"""Context from knowledge base:
{context}

---
Students Question: {question}

Pleasse answer the question based only on the context provided above."""
  response=groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[
          {"role":"system", "content":system_prompt},
          {"role":"user", "content":user_prompt}
      ],
      temperature=0.1,
      max_tokens=500
  )
  answer=response.choices[0].message.content
  return answer
print("RAG generation function defined.")

RAG generation function defined.


In [ ]:
def ask_college_assistant(question,top_k=3,verbose=True):
    if verbose:
        print(f"Question: {question}")
        print("="*60)
        print("Retreiving relevant documents...")
    results=retrieve_relavent_chunk(question,top_k=top_k)
    if verbose:
        print(f"Found {len(results['documents'][0])} documents")
    for i,meta in enumerate(results['metadatas'][0]):
        print(f"Document {i+1}:")
        print(f"  Subject: {meta['subject']}")
        print(f"  Topic: {meta['topic']}")
    context=build_context_from_results(results)
    if verbose:
        print(len(context))
        print("\nAnswer")
    return answer

In [ ]:
question_3="What is the popultation of Tokyo?"
answer_3=ask_college_assistant(question_3,top_k=3,verbose=True)
print(answer_3)


Question: What is the popultation of Tokyo?
Retreiving relevant documents...
Found 0 documents
0

Answer


NameError: name 'answer' is not defined

In [ ]:
def retreive_by_subject(Question,subject_filter,top_k=2):
    question_embedding=embedding_model.encode(question).tolist()
    results=collection.query(
        query_embeddings=question_embedding,
        n_results=top_k,
        where={"subject":subject_filter}
    )
    return results
    for i,doc

**Beginner questions**

1. What is hallucination in the context of LLMs?
2. What does RAG stand for? What problem does it solve
3. What is the role of vector DB in RAG pipeline

CODING QUESTIONS
1. Modify ask_college_assistabt ti display distance scores of retreived chunks

**MINI PROJECT **

dESCRIPTION:
Build a
complete knowledge assistant that:
* Looks the college_notes.csv knowledge base